# DC-Ops: Fine-tune YOLOv8n-seg on DC Component Labels

**If you just ran auto_label_colab.ipynb**, the labels and images are already in memory — skip to Cell 3.

**If starting fresh**, upload `dc_images.zip` and `dc_labels.zip` in Cell 2.

**Runtime → Change runtime type → T4 GPU**

In [ ]:
# 1. Install ultralytics
!pip install -q ultralytics

In [ ]:
# 2. Upload data (SKIP if continuing from auto_label_colab.ipynb)
import os, glob, zipfile

# Check if data already exists from labeling notebook
images_exist = len(glob.glob('data/sample_images/*.*')) > 0
labels_exist = len(glob.glob('labels/*.txt')) > 0

if images_exist and labels_exist:
    print(f'Data already loaded: {len(glob.glob("data/sample_images/*.*"))} images, {len(glob.glob("labels/*.txt"))} labels')
    print('Skipping upload.')
else:
    from google.colab import files
    print('Upload dc_images.zip and dc_labels.zip...')
    uploaded = files.upload()
    
    if 'dc_images.zip' in uploaded:
        with zipfile.ZipFile('dc_images.zip', 'r') as z:
            z.extractall('.')
    if 'dc_labels.zip' in uploaded:
        with zipfile.ZipFile('dc_labels.zip', 'r') as z:
            z.extractall('.')
    
    print(f'Images: {len(glob.glob("data/sample_images/*.*"))}')
    print(f'Labels: {len(glob.glob("labels/*.txt"))}')

In [ ]:
# 3. Prepare YOLO dataset structure
import shutil, random
from pathlib import Path
import yaml

DC_CLASSES = [
    'server rack', 'compute tray', 'NVLink switch tray', 'network switch',
    'power shelf', 'cable', 'network port', 'LED indicator',
    'label', 'fan', 'cooling manifold', 'cable cartridge',
    'power connector', 'drive bay', 'management port', 'DPU',
]

# Create dataset dirs
for split in ['train', 'val']:
    for sub in ['images', 'labels']:
        os.makedirs(f'dataset/{split}/{sub}', exist_ok=True)

# Find all images that have labels
images_dir = Path('data/sample_images')
labels_dir = Path('labels')

paired = []
for img in sorted(images_dir.iterdir()):
    if img.suffix.lower() in ('.jpg', '.jpeg', '.png', '.webp'):
        label = labels_dir / f'{img.stem}.txt'
        if label.exists() and label.stat().st_size > 0:
            paired.append((img, label))

random.seed(42)
random.shuffle(paired)
split_idx = int(len(paired) * 0.85)
train_set = paired[:split_idx]
val_set = paired[split_idx:]

for img, lbl in train_set:
    shutil.copy2(img, f'dataset/train/images/{img.name}')
    shutil.copy2(lbl, f'dataset/train/labels/{img.stem}.txt')

for img, lbl in val_set:
    shutil.copy2(img, f'dataset/val/images/{img.name}')
    shutil.copy2(lbl, f'dataset/val/labels/{img.stem}.txt')

# Create dataset.yaml
config = {
    'path': os.path.abspath('dataset'),
    'train': 'train/images',
    'val': 'val/images',
    'names': {i: name for i, name in enumerate(DC_CLASSES)},
}
with open('dataset/dataset.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print(f'Dataset ready: {len(train_set)} train, {len(val_set)} val')
print(f'Train labels: {len(os.listdir("dataset/train/labels"))}')
print(f'Val labels: {len(os.listdir("dataset/val/labels"))}')

In [ ]:
# 4. Train YOLOv8n-seg
from ultralytics import YOLO

model = YOLO('yolov8n-seg.pt')

results = model.train(
    data='dataset/dataset.yaml',
    epochs=50,
    batch=16,
    imgsz=640,
    project='runs',
    name='dc_ops_seg',
    exist_ok=True,
    device=0,          # T4 GPU
    patience=15,
    save=True,
    plots=True,
    verbose=True,
)

In [ ]:
# 5. Validate and show results
best_model = YOLO('runs/dc_ops_seg/weights/best.pt')
metrics = best_model.val()

print(f'\nmAP50 (box):  {metrics.box.map50:.3f}')
print(f'mAP50 (mask): {metrics.seg.map50:.3f}')
print(f'\nPer-class mAP50 (mask):')
for i, name in enumerate(DC_CLASSES):
    if i < len(metrics.seg.ap50):
        print(f'  {name:25s} {metrics.seg.ap50[i]:.3f}')

In [ ]:
# 6. Test on sample images
import matplotlib.pyplot as plt
import cv2
import numpy as np

val_images = sorted(glob.glob('dataset/val/images/*.*'))[:6]
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

for ax, img_path in zip(axes.flat, val_images):
    results = best_model(img_path, conf=0.3)
    annotated = results[0].plot()
    ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    ax.set_title(Path(img_path).name, fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# 7. Export and download trained model
import shutil

# Copy best weights
shutil.copy2('runs/dc_ops_seg/weights/best.pt', 'dc_ops_yolov8n_seg.pt')
shutil.copy2('runs/dc_ops_seg/weights/last.pt', 'dc_ops_yolov8n_seg_last.pt')

# Also export to ONNX (useful for verification)
best_model.export(format='onnx', imgsz=640, simplify=True)

import os
print(f'best.pt: {os.path.getsize("dc_ops_yolov8n_seg.pt") / 1e6:.1f} MB')
print(f'\nReady for download!')

In [ ]:
# 8. Download trained model
from google.colab import files
files.download('dc_ops_yolov8n_seg.pt')